<a href="https://colab.research.google.com/github/enilt/langgraph-alura/blob/main/Aula_01_ReAct_Groq_(V1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 01 - Agente ReAct com **Groq** (LLM)

Versão adaptada do notebook original da Alura (que usava o Google Gemini) para rodar com a **API da Groq**.

**Objetivos desta adaptação:**
- Trocar o LLM do Gemini para a **Groq** (modelos Llama / GPT-OSS, etc.).
- Ler a chave de API de forma **flexível**: funciona no **Google Colab** (via `userdata`) e no **VS Code** (via arquivo `.env`).
- Deixar o código **portable** para reaproveitar a lógica no **N8N local** no futuro (ver a última seção).

> A Groq expõe um endpoint compatível com OpenAI, então o mesmo modelo pode ser chamado tanto por código Python quanto pelos nós de IA do N8N.


## 1. Instalação das dependências

No **Colab**: rode a célula abaixo normalmente.

No **VS Code**: crie um ambiente virtual e instale os mesmos pacotes (pode rodar a célula também, ou usar o terminal):
```bash
python -m venv .venv
source .venv/bin/activate    # Windows: .venv\Scripts\activate
pip install groq python-dotenv langgraph
```


In [ ]:
# Groq (cliente oficial), dotenv para ler .env no VS Code, e langgraph (usado nas próximas aulas)
%pip install -U groq
%pip install -U python-dotenv
%pip install -U langgraph
# Opcional: integração Groq + LangChain (útil nas próximas aulas)
%pip install -U langchain-groq langchain-community


  Using cached groq-0.37.1-py3-none-any.whl.metadata (16 kB)
Using cached groq-0.37.1-py3-none-any.whl (137 kB)
  Attempting uninstall: groq
    Found existing installation: groq 1.7.0
    Uninstalling groq-1.7.0:
      Successfully uninstalled groq-1.7.0


## 2. Configuração da chave de API (Colab **e** VS Code)

A função `get_secret` abaixo procura a chave em 3 lugares, nesta ordem:
1. **Variável de ambiente** (`os.environ`) — útil em servidores / Docker / N8N.
2. **Google Colab** — menu lateral “Secrets” (ícone de chave). Adicione `GROQ_API_KEY`.
3. **Arquivo `.env`** — no VS Code, crie um arquivo `.env` na pasta do projeto:
   ```
   GROQ_API_KEY=gsk_sua_chave_aqui
   TAVILY_API_KEY=tvly_sua_chave_aqui   # usada nas próximas aulas (busca)
   ```

⚠️ Nunca comite o arquivo `.env` no Git. Crie sua chave gratuita em https://console.groq.com/keys


In [ ]:
import os

try:
    from dotenv import load_dotenv
    load_dotenv()  # carrega o .env quando rodando localmente (VS Code)
except Exception:
    pass


def get_secret(nome: str, obrigatorio: bool = True) -> str | None:
    """Busca um segredo em: os.environ -> Colab Secrets -> .env.

    Funciona igual no Colab e no VS Code, sem precisar mudar o código.
    """
    # 1) Variável de ambiente (inclui o que o .env carregou acima)
    valor = os.getenv(nome)
    if valor:
        return valor

    # 2) Google Colab (Secrets)
    try:
        from google.colab import userdata  # type: ignore
        valor = userdata.get(nome)
        if valor:
            os.environ[nome] = valor  # deixa disponível para o resto do notebook
            return valor
    except Exception:
        pass

    if obrigatorio:
        raise ValueError(
            f"Segredo '{nome}' não encontrado. "
            f"Defina no Colab (Secrets), no arquivo .env ou como variável de ambiente."
        )
    return None


# Carrega as chaves
GROQ_API_KEY = get_secret("GROQ_API_KEY")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# A TAVILY é usada nas aulas de busca (opcional aqui)
TAVILY_API_KEY = get_secret("TAVILY_API_KEY", obrigatorio=False)
if TAVILY_API_KEY:
    os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY

print("GROQ_API_KEY carregada:", "sim" if GROQ_API_KEY else "não")
print("TAVILY_API_KEY carregada:", "sim" if TAVILY_API_KEY else "não (opcional)")


GROQ_API_KEY carregada: sim
TAVILY_API_KEY carregada: não (opcional)


## 3. Inicialização do cliente Groq e teste rápido

Escolha do modelo: a Groq oferece vários modelos abertos e rápidos. Alguns bons padrões:
- `llama-3.3-70b-versatile` — forte e versátil (padrão aqui).
- `llama-3.1-8b-instant` — bem mais rápido/barato, bom para testes.
- `openai/gpt-oss-120b` — modelo grande, se disponível na sua conta.

Basta trocar a variável `MODELO_GROQ`.


In [ ]:
import re
from groq import Groq

# Modelo usado em todo o notebook. Troque aqui se quiser.
MODELO_GROQ = "openai/gpt-oss-120b"

client = Groq(api_key=GROQ_API_KEY)

# Teste rápido
resposta = client.chat.completions.create(
    model=MODELO_GROQ,
    temperature=0,
    messages=[{"role": "user", "content": "Diga apenas: Olá, mundo!"}],
)
print(resposta.choices[0].message.content)


Olá, mundo!


## 4. Um pequeno “wrapper” de chat para a Groq

O notebook original usava `model.start_chat()` do Gemini, que guardava o histórico automaticamente.
A Groq (API compatível com OpenAI) trabalha com uma **lista de mensagens**. A classe abaixo
reproduz o mesmo comportamento de forma simples — e ainda expõe `self.messages` para inspecionarmos o histórico.


In [ ]:
import json
from groq import BadRequestError


def _tool_call_para_texto_react(failed_generation: str) -> str:
    """Converte uma chamada de ferramenta nativa (JSON) para o formato de texto ReAct.

    Modelos da Groq (Llama 3.3/3.1, etc.) são treinados para "function calling" e,
    ao verem a descrição das ferramentas, às vezes respondem com um JSON
    {"name": ..., "arguments": {...}} em vez do texto "Ação: ...".
    Aqui convertemos esse JSON de volta para "Ação: nome: argumento".
    """
    try:
        data = json.loads(failed_generation)
    except Exception:
        return failed_generation
    nome = data.get("name", "")
    args = data.get("arguments", {}) or {}
    if args:
        arg = str(list(args.values())[0])
        return f"Ação: {nome}: {arg}"
    return f"Ação: {nome}"


class GroqChat:
    """Mantém o histórico da conversa e chama a Groq, imitando o start_chat() do Gemini.

    Tolera modelos que tentam chamar ferramentas de forma nativa: nesse caso,
    capturamos o erro 400 (tool_use_failed) e convertemos a chamada para o
    formato de texto ReAct, para o ciclo continuar normalmente.
    """

    def __init__(self, client, model, system=None, temperature=0):
        self.client = client
        self.model = model
        self.temperature = temperature
        self.messages = []
        if system:
            self.messages.append({"role": "system", "content": system})

    def send_message(self, content, stop=None):
        self.messages.append({"role": "user", "content": content})
        try:
            resp = self.client.chat.completions.create(
                model=self.model,
                temperature=self.temperature,
                messages=self.messages,
                stop=stop,  # para o modelo antes de inventar a "Observação"
            )
            texto = resp.choices[0].message.content or ""
        except BadRequestError as e:
            # O modelo tentou chamar uma ferramenta nativamente. Recuperamos a
            # chamada de dentro do erro e a convertemos para o texto "Ação: ...".
            body = getattr(e, "body", None) or {}
            erro = body.get("error", {}) if isinstance(body, dict) else {}
            failed = erro.get("failed_generation")
            if failed:
                texto = _tool_call_para_texto_react(failed)
            else:
                raise
        self.messages.append({"role": "assistant", "content": texto})
        return texto


## 5. Prompt ReAct (versão inicial)

Mesmo prompt do curso: o modelo raciocina em ciclos de **Pensamento → Ação → PAUSA → Observação → Resposta**.


In [ ]:
PROMPT_REACT = """
Você funciona em um ciclo de Pensamento, Ação, Pausa e Observação.
Ao final do ciclo, você fornece uma Resposta.
Use "Pensamento" para descrever seu raciocínio.
Use "Ação" para executar ferramentas - e então retorne "PAUSA".
A "Observação" será o resultado da ação executada.
Ações disponíveis:
  - consultar_estoque: retorna a quantidade disponível de um item no inventário (ex: "consultar_estoque: teclado")
  - consultar_preco_produto: retorna o preço unitário de um produto (ex: "consultar_preco_produto: mouse gamer")

Exemplo:
Pergunta: Quantos monitores temos em estoque?
Pensamento: Devo consultar a ação consultar_estoque para saber a quantidade de monitores.
Ação: consultar_estoque: monitor
PAUSA

Observação: Temos 75 monitores em estoque.
Resposta: Há 75 monitores em estoque.
""".strip()


## 6. Ferramentas (tools) do agente

In [ ]:
def consultar_estoque(item: str) -> str:
    """Simula a consulta de estoque de um item no inventário."""
    item = item.lower().strip()
    estoque = {
        "monitor": 75,
        "teclado": 120,
        "mouse gamer": 80,
        "webcam": 40,
        "headset": 60,
        "impressora": 15,
    }
    if item in estoque:
        return f"Temos {estoque[item]} {item}s em estoque."
    return f"Item '{item}' não encontrado no inventário."


def consultar_preco_produto(produto: str) -> str:
    """Simula a consulta do preço unitário de um produto."""
    produto = produto.lower().strip()
    precos = {
        "monitor": 999.90,
        "teclado": 150.00,
        "mouse gamer": 99.50,
        "webcam": 120.00,
        "headset": 180.00,
        "impressora": 750.00,
    }
    if produto in precos:
        return f"O preço de um(a) {produto} é R$ {precos[produto]:.2f}."
    return f"Produto '{produto}' não encontrado na lista de preços."


In [ ]:
print(consultar_estoque("teclado"))
print(consultar_preco_produto("impressora"))
print(consultar_estoque("monitor"))


Temos 120 teclados em estoque.
O preço de um(a) impressora é R$ 750.00.
Temos 75 monitors em estoque.


## 7. O loop ReAct usando a Groq

Diferenças em relação ao Gemini:
- Usamos o `GroqChat` (lista de mensagens) no lugar de `model.start_chat()`.
- Passamos `stop=["PAUSA", "Observação:"]` para o modelo **parar** logo após escolher a ação,
  evitando que ele invente a observação sozinho.


In [ ]:
def run_react_agent(pergunta: str, max_iterations: int = 5) -> str:
    chat = GroqChat(client, MODELO_GROQ, system=PROMPT_REACT, temperature=0)
    current_prompt = pergunta

    for i in range(max_iterations):
        response_text = chat.send_message(
            current_prompt, stop=["PAUSA", "Observação:"]
        ).strip()

        print(f"--- Iteração {i+1} ---")
        print(f"Modelo pensou/respondeu:\n{response_text}\n")

        # Resposta final?
        resp_final = re.search(r"Resposta:\s*(.*)", response_text, re.DOTALL)
        if resp_final:
            return resp_final.group(1).strip()

        # Extrai a ação (com ou sem argumento)
        match = re.search(r"Ação:\s*(\w+)(?::\s*([^\n]*))?", response_text)
        if match:
            action_name = match.group(1).strip()
            action_arg = match.group(2).strip() if match.group(2) is not None else ""

            if action_name == "consultar_estoque":
                observacao = consultar_estoque(action_arg)
            elif action_name == "consultar_preco_produto":
                observacao = consultar_preco_produto(action_arg)
            else:
                observacao = f"Erro: Ação '{action_name}' desconhecida."

            current_prompt = f"Observação: {observacao}"
            print(f"Executou ação: {action_name}('{action_arg}')")
            print(f"Observação: {observacao}\n")
        else:
            return (
                f"Erro: não consegui extrair uma Ação ou Resposta final "
                f"após {i+1} iterações. Última resposta: {response_text}"
            )

    return "Erro: Limite máximo de iterações atingido sem uma resposta final."


In [ ]:
pergunta_1 = "Quantos mouses gamers estão no inventário?"
print(f"**Interação 1: {pergunta_1}**")
resposta_1 = run_react_agent(pergunta_1)
print(f"\n**RESPOSTA FINAL DO AGENTE 1:** {resposta_1}\n")
print("\n" + "=" * 50 + "\n")


**Interação 1: Quantos mouses gamers estão no inventário?**
--- Iteração 1 ---
Modelo pensou/respondeu:
Pensamento: Preciso consultar o estoque para saber a quantidade de mouses gamers disponíveis.  
Ação: consultar_estoque: mouse gamer

Executou ação: consultar_estoque('mouse gamer')
Observação: Temos 80 mouse gamers em estoque.

--- Iteração 2 ---
Modelo pensou/respondeu:
Resposta: Há 80 mouses gamers no inventário.


**RESPOSTA FINAL DO AGENTE 1:** Há 80 mouses gamers no inventário.





In [ ]:
pergunta_2 = "Qual o valor de uma impressora?"
print(f"**Interação 2: {pergunta_2}**")
resposta_2 = run_react_agent(pergunta_2)
print(f"\n**RESPOSTA FINAL DO AGENTE 2:** {resposta_2}\n")


**Interação 2: Qual o valor de uma impressora?**
--- Iteração 1 ---
Modelo pensou/respondeu:



**RESPOSTA FINAL DO AGENTE 2:** Erro: não consegui extrair uma Ação ou Resposta final após 1 iterações. Última resposta: 



In [ ]:
pergunta_3 = "Temos cadeiras em estoque?"
print(f"**Interação 3: {pergunta_3}**")
resposta_3 = run_react_agent(pergunta_3)
print(f"\n**RESPOSTA FINAL DO AGENTE 3:** {resposta_3}\n")


**Interação 3: Temos cadeiras em estoque?**
--- Iteração 1 ---
Modelo pensou/respondeu:



**RESPOSTA FINAL DO AGENTE 3:** Erro: não consegui extrair uma Ação ou Resposta final após 1 iterações. Última resposta: 



## 8. Adicionando novas ferramentas

Como no curso, vamos evoluir o agente: uma ferramenta para achar o **produto mais caro**
e outra para **calcular o valor total** de uma lista de itens.


In [ ]:
PRECOS_INVENTARIO = {
    "monitor": 999.90,
    "teclado": 150.00,
    "mouse gamer": 99.50,
    "webcam": 120.00,
    "headset": 180.00,
    "impressora": 750.00,
}


def ferramenta_encontrar_produto_mais_caro() -> str:
    """Retorna o nome e o preço do produto mais caro. Não precisa de argumentos."""
    if not PRECOS_INVENTARIO:
        return "Nenhum produto encontrado na lista de preços para comparação."
    nome = max(PRECOS_INVENTARIO, key=PRECOS_INVENTARIO.get)
    valor = PRECOS_INVENTARIO[nome]
    return f"O produto mais caro é o(a) {nome} com preço de R$ {valor:.2f}."


def ferramenta_calcular_valor_total_lista(lista_itens: str) -> str:
    """Calcula o valor total de uma lista de itens separados por vírgula."""
    itens = [i.strip().lower() for i in lista_itens.split(",")]
    total = 0.0
    nao_encontrados = []
    for item in itens:
        if item in PRECOS_INVENTARIO:
            total += PRECOS_INVENTARIO[item]
        else:
            nao_encontrados.append(item)
    resposta = f"O valor total dos itens encontrados é R$ {total:.2f}."
    if nao_encontrados:
        resposta += (
            f" Os seguintes itens não foram encontrados e não entraram no cálculo: "
            f"{', '.join(nao_encontrados)}."
        )
    return resposta


## 9. Prompt ReAct completo (com todas as ferramentas)

In [ ]:
PROMPT_REACT = """
Você funciona em um ciclo de Pensamento, Ação, Pausa e Observação.
Ao final do ciclo, você fornece uma Resposta.
Use "Pensamento" para descrever seu raciocínio.
Use "Ação" para executar ferramentas - e então retorne "PAUSA".
A "Observação" será o resultado da ação executada.
Ações disponíveis:
  - consultar_estoque: retorna a quantidade disponível de um item no inventário (ex: "consultar_estoque: teclado")
  - consultar_preco_produto: retorna o preço unitário de um produto (ex: "consultar_preco_produto: mouse gamer")
  - encontrar_produto_mais_caro: retorna o nome e o preço do produto mais caro no inventário (não requer argumentos)
  - calcular_valor_total_lista: calcula o valor total de uma lista de itens de compra. Recebe uma string com itens separados por vírgula (ex: "teclado, mouse gamer, monitor")

Exemplo:
Pergunta: Quantos monitores temos em estoque?
Pensamento: Devo consultar a ação consultar_estoque para saber a quantidade de monitores.
Ação: consultar_estoque: monitor
PAUSA

Observação: Temos 75 monitores em estoque.
Resposta: Há 75 monitores em estoque.

Exemplo:
Pergunta: Qual é o produto mais caro?
Pensamento: Preciso usar a ação encontrar_produto_mais_caro.
Ação: encontrar_produto_mais_caro
PAUSA

Observação: O produto mais caro é o(a) monitor com preço de R$ 999.90.
Resposta: O produto mais caro é o monitor, custando R$ 999.90.

Exemplo:
Pergunta: Quanto custa um teclado e um mouse gamer?
Pensamento: O usuário quer o valor total de vários itens. Devo usar calcular_valor_total_lista.
Ação: calcular_valor_total_lista: teclado, mouse gamer
PAUSA

Observação: O valor total dos itens encontrados é R$ 249.50.
Resposta: O valor total do teclado e do mouse gamer é R$ 249.50.
""".strip()


## 10. Loop ReAct atualizado (roteia todas as ferramentas)

In [ ]:
def run_react_agent(pergunta: str, max_iterations: int = 5) -> str:
    chat = GroqChat(client, MODELO_GROQ, system=PROMPT_REACT, temperature=0)
    current_prompt = pergunta

    for i in range(max_iterations):
        response_text = chat.send_message(
            current_prompt, stop=["PAUSA", "Observação:"]
        ).strip()

        print(f"\n--- Iteração {i+1} ---")
        print(f"Modelo pensou/respondeu:\n{response_text}\n")

        resp_final = re.search(r"Resposta:\s*(.*)", response_text, re.DOTALL)
        if resp_final:
            return resp_final.group(1).strip()

        match = re.search(r"Ação:\s*(\w+)(?::\s*([^\n]*))?", response_text)
        if match:
            action_name = match.group(1).strip()
            action_arg = match.group(2).strip() if match.group(2) is not None else ""

            if action_name == "consultar_estoque":
                observacao = consultar_estoque(action_arg)
            elif action_name == "consultar_preco_produto":
                observacao = consultar_preco_produto(action_arg)
            elif action_name == "encontrar_produto_mais_caro":
                observacao = ferramenta_encontrar_produto_mais_caro()
            elif action_name == "calcular_valor_total_lista":
                observacao = ferramenta_calcular_valor_total_lista(action_arg)
            else:
                observacao = f"Erro: Ação '{action_name}' desconhecida."

            current_prompt = f"Observação: {observacao}"
            print(f"Executou ação: {action_name} com argumento '{action_arg}'")
            print(f"Observação: {observacao}\n")
        else:
            return (
                f"Erro: não consegui extrair uma Ação ou Resposta final "
                f"após {i+1} iterações. Última resposta: {response_text}"
            )

    return "Erro: Limite máximo de iterações atingido sem uma resposta final."


In [ ]:
print("--- Começando as Interações com o Agente ReAct ---")

for pergunta in [
    "Quantos teclados temos em estoque?",
    "Qual o preço de um headset?",
    "Temos cadeiras em estoque?",
    "Qual é o produto mais caro?",
    "Qual o valor de um teclado, uma impressora e uma webcam?",
]:
    print(f"\n**Pergunta: {pergunta}**")
    resposta = run_react_agent(pergunta)
    print(f"\n**RESPOSTA FINAL:** {resposta}\n")
    print("=" * 80)

print("\n--- Fim das Interações ---")


--- Começando as Interações com o Agente ReAct ---

**Pergunta: Quantos teclados temos em estoque?**

--- Iteração 1 ---
Modelo pensou/respondeu:



**RESPOSTA FINAL:** Erro: não consegui extrair uma Ação ou Resposta final após 1 iterações. Última resposta: 


**Pergunta: Qual o preço de um headset?**

--- Iteração 1 ---
Modelo pensou/respondeu:



**RESPOSTA FINAL:** Erro: não consegui extrair uma Ação ou Resposta final após 1 iterações. Última resposta: 


**Pergunta: Temos cadeiras em estoque?**

--- Iteração 1 ---
Modelo pensou/respondeu:



**RESPOSTA FINAL:** Erro: não consegui extrair uma Ação ou Resposta final após 1 iterações. Última resposta: 


**Pergunta: Qual é o produto mais caro?**

--- Iteração 1 ---
Modelo pensou/respondeu:



**RESPOSTA FINAL:** Erro: não consegui extrair uma Ação ou Resposta final após 1 iterações. Última resposta: 


**Pergunta: Qual o valor de um teclado, uma impressora e uma webcam?**

--- Iteração 1 ---
Modelo pensou/respondeu:



**RESPOSTA FINA

## 11. Versão que retorna o histórico completo

Útil para depurar o raciocínio do agente. Com a Groq, o histórico já é a própria `chat.messages`.


In [ ]:
def run_react_agent_with_history(pergunta: str, max_iterations: int = 5):
    chat = GroqChat(client, MODELO_GROQ, system=PROMPT_REACT, temperature=0)
    current_prompt = pergunta

    for i in range(max_iterations):
        response_text = chat.send_message(
            current_prompt, stop=["PAUSA", "Observação:"]
        ).strip()

        resp_final = re.search(r"Resposta:\s*(.*)", response_text, re.DOTALL)
        if resp_final:
            return resp_final.group(1).strip(), chat.messages

        match = re.search(r"Ação:\s*(\w+)(?::\s*([^\n]*))?", response_text)
        if match:
            action_name = match.group(1).strip()
            action_arg = match.group(2).strip() if match.group(2) is not None else ""
            if action_name == "consultar_estoque":
                observacao = consultar_estoque(action_arg)
            elif action_name == "consultar_preco_produto":
                observacao = consultar_preco_produto(action_arg)
            elif action_name == "encontrar_produto_mais_caro":
                observacao = ferramenta_encontrar_produto_mais_caro()
            elif action_name == "calcular_valor_total_lista":
                observacao = ferramenta_calcular_valor_total_lista(action_arg)
            else:
                observacao = f"Erro: Ação '{action_name}' desconhecida."
            current_prompt = f"Observação: {observacao}"
        else:
            return (f"Erro após {i+1} iterações: {response_text}", chat.messages)

    return "Erro: limite de iterações atingido.", chat.messages


In [ ]:
resposta_exemplo, historico = run_react_agent_with_history("Quantos teclados temos em estoque?")
print(f"**RESPOSTA FINAL:** {resposta_exemplo}\n")
print("--- Histórico Completo da Interação ---")
for i, msg in enumerate(historico):
    print(f"--- Mensagem {i+1} (role: {msg['role']}) ---")
    print(msg["content"])
    print("-" * 20)
print("\n--- Fim do Histórico ---")


**RESPOSTA FINAL:** Erro após 1 iterações: 

--- Histórico Completo da Interação ---
--- Mensagem 1 (role: system) ---
Você funciona em um ciclo de Pensamento, Ação, Pausa e Observação.
Ao final do ciclo, você fornece uma Resposta.
Use "Pensamento" para descrever seu raciocínio.
Use "Ação" para executar ferramentas - e então retorne "PAUSA".
A "Observação" será o resultado da ação executada.
Ações disponíveis:
  - consultar_estoque: retorna a quantidade disponível de um item no inventário (ex: "consultar_estoque: teclado")
  - consultar_preco_produto: retorna o preço unitário de um produto (ex: "consultar_preco_produto: mouse gamer")
  - encontrar_produto_mais_caro: retorna o nome e o preço do produto mais caro no inventário (não requer argumentos)
  - calcular_valor_total_lista: calcula o valor total de uma lista de itens de compra. Recebe uma string com itens separados por vírgula (ex: "teclado, mouse gamer, monitor")

Exemplo:
Pergunta: Quantos monitores temos em estoque?
Pensamento

## 12. Modo conversa (interativo)

No **VS Code** e no **Jupyter** o `input()` funciona normalmente. No Colab também funciona.
Digite `sair` para encerrar.


In [ ]:
def iniciar_conversacao_com_agente():
    print("--- Agente de Inventário Interativo (Groq) ---")
    print("Digite sua pergunta sobre o inventário, ou digite 'sair' para encerrar.")
    print("-" * 50)
    while True:
        pergunta_usuario = input("\nVocê: ")
        if pergunta_usuario.lower().strip() == "sair":
            print("Encerrando a conversa. Até logo!")
            break
        print("\nAgente: Processando...")
        try:
            resposta_agente = run_react_agent(pergunta_usuario)
            print(f"\nAgente: {resposta_agente}")
        except Exception as e:
            print(f"\nAgente: Ocorreu um erro: {e}")
            print("Tente novamente ou digite 'sair'.")


# Descomente para rodar o modo interativo:
# iniciar_conversacao_com_agente()


## 13. Bônus: reaproveitando no **N8N local** (futuro)

Quando quiser levar essa lógica para um fluxo do **N8N** rodando localmente, você tem 2 caminhos.
A Groq é **compatível com a API da OpenAI**, o que facilita muito.

### Caminho A — Nós de IA nativos do N8N (recomendado)
1. Adicione as credenciais da Groq no N8N:
   - Use a credencial **"OpenAI"** (ou "Groq", nas versões recentes) e informe:
     - **Base URL / API endpoint**: `https://api.groq.com/openai/v1`
     - **API Key**: sua `GROQ_API_KEY`.
2. Use o nó **"AI Agent"** (LangChain) com um **Chat Model** apontando para a Groq
   e escolha o modelo (ex.: `llama-3.3-70b-versatile`).
3. Cada ferramenta Python deste notebook (`consultar_estoque`, `consultar_preco_produto`, etc.)
   vira uma **Tool** no N8N:
   - Um nó **"Custom Code Tool"** (JavaScript) para lógica simples, ou
   - Um nó **"HTTP Request Tool"** chamando um endpoint seu (ver Caminho B).

### Caminho B — Expor as ferramentas como uma API e chamar via HTTP Request
Se preferir manter a lógica em Python, exponha as ferramentas como um pequeno serviço local
e no N8N use o nó **HTTP Request**. Exemplo mínimo com FastAPI (rode você mesmo, no seu VS Code):

```python
# ferramentas_api.py  (rode com: uvicorn ferramentas_api:app --host 127.0.0.1 --port 8000)
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

PRECOS = {"monitor": 999.90, "teclado": 150.0, "mouse gamer": 99.5,
          "webcam": 120.0, "headset": 180.0, "impressora": 750.0}
ESTOQUE = {"monitor": 75, "teclado": 120, "mouse gamer": 80,
           "webcam": 40, "headset": 60, "impressora": 15}

class ItemIn(BaseModel):
    item: str

@app.post("/consultar_estoque")
def estoque(dados: ItemIn):
    k = dados.item.lower().strip()
    return {"resultado": f"Temos {ESTOQUE[k]} {k}s em estoque." if k in ESTOQUE
            else f"Item '{k}' não encontrado."}

@app.post("/consultar_preco")
def preco(dados: ItemIn):
    k = dados.item.lower().strip()
    return {"resultado": f"O preço de {k} é R$ {PRECOS[k]:.2f}." if k in PRECOS
            else f"Produto '{k}' não encontrado."}
```

No N8N (rodando local, ex.: via Docker) você aponta o nó **HTTP Request** para
`http://host.docker.internal:8000/consultar_estoque` (Docker) ou `http://127.0.0.1:8000/...`
e conecta como **Tool** do nó AI Agent.

> Observação de segurança: mantenha o serviço ouvindo apenas em `127.0.0.1` (local) e não
> exponha a chave da Groq no fluxo — guarde-a nas **Credentials** do N8N.

### Resumo
- **Colab / VS Code (agora):** este notebook, com a chave lida via `get_secret`.
- **N8N (futuro):** mesma chave Groq via credencial OpenAI-compatível; ferramentas viram nós Tool.
